In [ ]:
def is_running_in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_running_in_colab():
    %cd /content/drive/MyDrive/02_tcn_fx_predictor/src

/content/drive/MyDrive/02_tcn_fx_predictor/src


In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import weight_norm
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# @title
# TCN
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    # exclude some data from -chomp_size to the end
    # in order to prohibit information leakage
    # padding is required since sequence length cannot change
    # but padding works on both left and right sides
    # e.g. when k=3, padding_left = padding_right=1,
    # at t=0, receptive field covers t=-1,0,1, which incurs leakage
    def forward(self, x):
        # contiguous() keep the memory contiguous
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super().__init__()
        # n_inputs: num of features, n_outputs: output channels
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size=kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv1 = weight_norm(self.conv1)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size=kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.conv2.weight.data.normal_(0, 0.01)
        self.conv2 = weight_norm(self.conv2)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2
        )
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels): # how many layers
            # i=0,1,2...
            dilation = 2**i
            # num_inputs: dim of inputs
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [
                TemporalBlock(in_channels, out_channels, kernel_size=kernel_size,
                            stride=1, dilation=dilation,
                            padding=(kernel_size-1)*dilation, dropout=dropout)
            # temporal block = 2*( chomp1d + conv1d + relu + dropout)
            ]
        # *layers: unpack layers
        self.networks = nn.Sequential(*layers)
        # input dim: num_channels[-1]
        self.fc = nn.Linear(num_channels[-1], 1)

    def forward(self, x):
        x = self.networks(x)
        # (bs, num_channels, sequence length) -> (bs, num_channels)
        x = x[:, :, -1]
        x = self.fc(x)
        return x


def generate_sine_waves(sequence_length, n_samples, noise=0.1):
    t = np.linspace(0, 100, sequence_length + n_samples)
    data = np.sin(t) + 0.5*np.sin(2*t) + noise*np.random.randn(sequence_length + n_samples)
    X, y = [], []
    for i in range(n_samples):
        X.append(data[i:i+sequence_length])
        y.append(data[i+sequence_length])
    return np.array(X), np.array(y)

def prepare_data(X, y, batch_size=64, train_ratio=0.8):
    # X 的形状是 (n_samples, sequence_length)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=train_ratio, shuffle=False
    )

    # 标准化
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

    # 转换为 PyTorch 张量
    X_train = torch.FloatTensor(X_train_scaled).unsqueeze(1)  # (batch, 1, seq_len)
    X_test = torch.FloatTensor(X_test_scaled).unsqueeze(1)
    y_train = torch.FloatTensor(y_train_scaled).unsqueeze(1)  # (batch, 1)
    y_test = torch.FloatTensor(y_test_scaled).unsqueeze(1)

    train_dataset = TensorDataset(X_train, y_train)
    test_dataset = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader, scaler_X, scaler_y

def train_model(model, train_loader, test_loader, epochs=50, lr=0.01, device='cpu'):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    train_loss, test_loss = [], []

    for ep in range(epochs):
        epoch_train_loss = 0.0
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item()

        model.eval()
        epoch_test_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                epoch_test_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_test_loss = epoch_test_loss / len(test_loader)
        train_loss.append(avg_train_loss)
        test_loss.append(avg_test_loss)

        scheduler.step(avg_test_loss)

        if (ep + 1) % 10 == 0:
            print(f'Epoch {ep+1}: Train Loss = {avg_train_loss:.6f}, Test Loss = {avg_test_loss:.6f}')

    return train_loss, test_loss

# 主程序
if __name__ == "__main__":
    SEQ_LENGTH = 50
    BATCH_SIZE = 64
    EPOCHS = 50
    LEARNING_RATE = 0.001
    DROPOUT = 0.2
    NUM_CHANNELS = [32, 64, 32]  # 移除了最后的1
    KERNEL_SIZE = 3
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 生成数据
    X, y = generate_sine_waves(SEQ_LENGTH, 3000)
    print(f"Data shape: X={X.shape}, y={y.shape}")

    # 准备数据
    train_loader, test_loader, scaler_X, scaler_y = prepare_data(
        X, y, batch_size=BATCH_SIZE
    )

    # 创建模型
    model = TemporalConvNet(
        num_inputs=1,
        num_channels=NUM_CHANNELS,
        kernel_size=KERNEL_SIZE,
        dropout=DROPOUT
    )

    # 训练模型
    train_losses, test_losses = train_model(
        model, train_loader, test_loader,
        epochs=EPOCHS, lr=LEARNING_RATE, device=device
    )

    # 绘制损失曲线
    plt.figure(figsize=(10, 4))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Training History')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

Data shape: X=(3000, 50), y=(3000,)


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


KeyboardInterrupt: 

In [ ]:
!pip install dukascopy-python lightning torchinfo optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.4/270.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 62.1 MB/s eta 0:00:00


In [ ]:
def is_running_in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_running_in_colab():
    %cd /content/drive/MyDrive/02_tcn_fx_predictor/src

import sys
import os
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import datetime as dt
import dukascopy_python
from dukascopy_python.instruments import INSTRUMENT_FX_MAJORS_EUR_USD

import polars as pl
from polars import col as c
import pandas as pd
import numpy as np
from src.configs.configs import BASE_DIR, DATA_DIR, DATA_DIR_RAW, DATA_DIR_PROCESSED, DATA_DIR_FEATURES_TARGETS
from collections import defaultdict

import torch
import torch.nn as nn
import lightning as L
from torchinfo import summary
from torch.utils.data import TensorDataset, DataLoader
from src.models.TCN import TemporalConvNet, LightningTCN
from sklearn.preprocessing import StandardScaler
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
import optuna
#from optuna.integration import PyTorchLightningPruningCallback

/content/drive/MyDrive/02_tcn_fx_predictor/src


In [ ]:
# @title
trading_days = pd.bdate_range(start='2025-01-02', end='2025-01-31')
last_day = trading_days[-1] + pd.Timedelta(days=1)
trading_days = trading_days.union(pd.DatetimeIndex([last_day]))
print(trading_days)

s = 0
data_list=[]
while True:
    e = min( s+1, len(trading_days)-1)
    start = trading_days[s]
    end = trading_days[e]

    data = dukascopy_python.fetch(
        instrument=INSTRUMENT_FX_MAJORS_EUR_USD,
        interval=dukascopy_python.INTERVAL_TICK,
        offer_side=dukascopy_python.OFFER_SIDE_BID,
        start=start,
        end=end,
)
    if not data.empty:
        data = data.reset_index()
        data_list.append( pl.from_pandas(data).lazy() )
    s = e
    if e == len(trading_days)-1:
        break


lf = pl.concat(data_list, how='vertical')
lf.sink_parquet(DATA_DIR_RAW/'raw_data.pqrquet')
print(lf.collect().head(10))

DatetimeIndex(['2025-01-02', '2025-01-03', '2025-01-06', '2025-01-07',
               '2025-01-08', '2025-01-09', '2025-01-10', '2025-01-13',
               '2025-01-14', '2025-01-15', '2025-01-16', '2025-01-17',
               '2025-01-20', '2025-01-21', '2025-01-22', '2025-01-23',
               '2025-01-24', '2025-01-27', '2025-01-28', '2025-01-29',
               '2025-01-30', '2025-01-31', '2025-02-01'],
              dtype='datetime64[ns]', freq=None)


INFO:DUKASCRIPT:current timestamp :2025-01-02T10:28:15.211000
INFO:DUKASCRIPT:current timestamp :2025-01-02T14:58:38.019000
INFO:DUKASCRIPT:current timestamp :2025-01-02T17:45:41.797000
INFO:DUKASCRIPT:current timestamp :2025-01-03T12:18:30.685000
INFO:DUKASCRIPT:current timestamp :2025-01-03T18:15:38.656000
INFO:DUKASCRIPT:current timestamp :2025-01-06T10:18:34.969000
INFO:DUKASCRIPT:current timestamp :2025-01-06T13:22:00.679000
INFO:DUKASCRIPT:current timestamp :2025-01-06T15:36:29.373000
INFO:DUKASCRIPT:current timestamp :2025-01-07T10:06:07.314000
INFO:DUKASCRIPT:current timestamp :2025-01-07T15:06:46.539000
INFO:DUKASCRIPT:current timestamp :2025-01-07T18:19:49.701000
INFO:DUKASCRIPT:current timestamp :2025-01-08T10:15:09.851000
INFO:DUKASCRIPT:current timestamp :2025-01-08T13:35:18.990000
INFO:DUKASCRIPT:current timestamp :2025-01-08T17:10:21.008000
INFO:DUKASCRIPT:current timestamp :2025-01-09T09:26:46.757000
INFO:DUKASCRIPT:current timestamp :2025-01-09T15:50:05.877000
INFO:DUK

shape: (10, 5)
┌─────────────────────────────┬──────────┬──────────┬───────────┬───────────┐
│ timestamp                   ┆ bidPrice ┆ askPrice ┆ bidVolume ┆ askVolume │
│ ---                         ┆ ---      ┆ ---      ┆ ---       ┆ ---       │
│ datetime[ns, UTC]           ┆ f64      ┆ f64      ┆ f64       ┆ f64       │
╞═════════════════════════════╪══════════╪══════════╪═══════════╪═══════════╡
│ 2025-01-02 00:00:00.848 UTC ┆ 1.03508  ┆ 1.03511  ┆ 0.9       ┆ 0.9       │
│ 2025-01-02 00:00:00.951 UTC ┆ 1.03507  ┆ 1.03512  ┆ 6.75      ┆ 0.9       │
│ 2025-01-02 00:00:01.469 UTC ┆ 1.0351   ┆ 1.03512  ┆ 0.9       ┆ 0.45      │
│ 2025-01-02 00:00:01.574 UTC ┆ 1.0351   ┆ 1.03513  ┆ 0.9       ┆ 0.9       │
│ 2025-01-02 00:00:01.627 UTC ┆ 1.0351   ┆ 1.03515  ┆ 0.9       ┆ 0.9       │
│ 2025-01-02 00:00:02.367 UTC ┆ 1.03511  ┆ 1.03516  ┆ 1.8       ┆ 6.39      │
│ 2025-01-02 00:00:03.134 UTC ┆ 1.0351   ┆ 1.03516  ┆ 1.89      ┆ 5.49      │
│ 2025-01-02 00:00:03.238 UTC ┆ 1.0351   ┆ 1.0351

In [ ]:
# @title
lf = pl.scan_parquet(DATA_DIR_RAW/'raw_data.pqrquet').fill_null(strategy='forward').fill_null(strategy='backward').fill_null(0)
lf = lf.sort('timestamp').with_columns(
    c.timestamp.dt.truncate('10s'),
    micro_price = (c.askVolume * c.bidPrice + c.askPrice*c.bidVolume)/(c.bidVolume + c.askVolume)
      ).group_by('timestamp').agg(
          pl.all().last(),
          ask_min=c.askPrice.min(),
          bid_min=c.bidPrice.min(),
          ask_max=c.askPrice.max(),
          bid_max=c.bidPrice.max(),
          ask_std=c.askPrice.std(),
          bid_std=c.bidPrice.std()
          ).sort('timestamp')
lf = lf.collect().upsample('timestamp', every='10s').with_columns(
    pl.all().exclude('timestamp').forward_fill()
).lazy()
lf = lf.filter(c.timestamp.dt.weekday() <= 5)


#————————————————————————————————————————
lf = lf.with_columns(

    rolling_mean_10s = c.micro_price.rolling_mean(window_size=10),
    rolling_mean_30s = c.micro_price.rolling_mean(window_size=30),
    rolling_mean_60s = c.micro_price.rolling_mean(window_size=60),
    rolling_mean_180s = c.micro_price.rolling_mean(window_size=180),
)
lf = lf.with_columns(
    h10 = c.rolling_mean_10s.shift(-10),
    h30 = c.rolling_mean_30s.shift(-30),
    h60 = c.rolling_mean_60s.shift(-60),
    h180 = c.rolling_mean_180s.shift(-180),
)

lf = lf.with_columns(
    d10 = c.h10.log() - c.rolling_mean_10s.log(),
    d30 = c.h30.log() - c.rolling_mean_30s.log(),
    d60 = c.h60.log() - c.rolling_mean_60s.log(),
    d180 = c.h180.log() - c.rolling_mean_180s.log(),

)
# ——————————————-——————————————————————
cols = [f'd{n}' for n in [10, 30, 60, 180]]
lf = lf.drop_nulls()
df = lf.collect()

cutoff = int( df.height * 0.8 * 0.8 )
df = df.slice(0,cutoff)
quantiles = defaultdict(dict)

for cl in cols:
    quantiles[cl]['.01_q'] = df.select(cl).quantile(0.01).item()
    quantiles[cl]['.4_q'] = df.select(cl).quantile(0.4).item()
    quantiles[cl]['.6_q'] = df.select(cl).quantile(0.6).item()
    quantiles[cl]['.99_q'] = df.select(cl).quantile(0.99).item()


#——————————————————————————————————
lf = lf.with_columns(
    [ pl.col(cl).clip(quantiles[cl]['.01_q'], quantiles[cl]['.99_q'] ) for cl in cols
      ]
)
thrd = defaultdict(dict)
for cl in cols:
    thrd[cl]['threshold'] = lf.select( pl.col(cl).abs()).quantile(0.1).collect().item()

lf = lf.with_columns(
    [ (pl.when( pl.col(cl)>thrd[cl]['threshold'] ).then(2).when(pl.col(cl)<-1*thrd[cl]['threshold']).then(0).otherwise(1)).alias(cl) for cl in cols
      ]
)
cols2drop = [f'h{n}' for n in [10,30,60,180]] +[f'rolling_mean_{n}s' for n in [10,30,60,180]]
lf = lf.drop(cols2drop).sink_parquet(DATA_DIR_PROCESSED/'dataset_with_targets.parquet')



In [ ]:
data = pl.scan_parquet(DATA_DIR_PROCESSED/'dataset_with_targets.parquet')
data = data.with_columns(
    order_imbalance = c.bidVolume.log()-c.askVolume.log(),
    total_depth = c.bidVolume + c.askVolume,
    delta_bid_volume = c.bidVolume.diff(),
    delta_ask_volume = c.askVolume.diff(),
    micro_ret = c.micro_price.log() - c.micro_price.shift(1).log()
)

data =data.with_columns(
    rolling_ret_mean_5 = c.micro_ret.rolling_mean(window_size=5),
    rolling_ret_mean_10 = c.micro_ret.rolling_mean(window_size=10),
    rolling_ret_mean_50 = c.micro_ret.rolling_mean(window_size=50),
)

data.fill_null(strategy='backward').drop_nulls().sink_parquet(DATA_DIR_FEATURES_TARGETS/'features_targets')

In [ ]:
sql=48
def sliding_window(X, sql):
  X_list = []
  for i in range(X.shape[0]-sql+1):
      X_list.append(X[i:i+sql, :])
  X = np.stack(X_list, axis=0)
  return X



data = pl.scan_parquet(DATA_DIR_FEATURES_TARGETS/'features_targets').sort('timestamp')
target_cols = [f'd{n}' for n in [10,30,60,180]]
X = data.select(pl.all().exclude(target_cols+['timestamp'])).collect().to_numpy()
y = data.select(target_cols).collect().to_numpy()
cutoff1 = int( y.shape[0] * 0.8 *0.8 )
cutoff2 = int( y.shape[0] *0.8 )
X_train = X[:cutoff1]
y_train = y[:cutoff1]
X_val = X[cutoff1:cutoff2][180:]
y_val = y[cutoff1:cutoff2][180:]
X_test = X[cutoff2:][180:]
y_test = y[cutoff2:][180:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


X_train = sliding_window(X_train, sql).transpose(0,2,1)
y_train = y_train[sql-1:,:]
X_val = sliding_window(X_val, sql).transpose(0,2,1)
y_val = y_val[sql-1:,:]
X_test = sliding_window(X_test, sql).transpose(0,2,1)
y_test = y_test[sql-1:,:]
print(X_train.shape)
print(y_train.shape)


(120913, 19, 48)
(120913, 4)


In [ ]:
class LightningTCN(L.LightningModule):
    def __init__(self, model, lr, criterion, optimizer_):
        super().__init__()
        self.model = model
        self.lr =lr
        self.criterion = criterion
        self.optimizer_ = optimizer_

    def forward(self,x):
        return self.model(x)


    def training_step(self, batch, batch_idx):
        x,y=batch # unpack
        logits = self(x)
        loss = self.criterion(logits, y)

        preds = torch.argmax(logits, axis=1)
        acc = (y==preds).float().mean()

        self.log('train loss', loss, prog_bar=True, on_epoch=True)
        self.log('train accuracy', acc, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x,y=batch # unpack
        logits = self(x)
        loss = self.criterion(logits, y)

        preds = torch.argmax(logits, axis=1)
        acc = (y==preds).float().mean()

        self.log('val loss', loss, prog_bar=True, on_epoch=True)
        self.log('val accuracy', acc, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):
        optimizer = self.optimizer_(self.model.parameters(), lr=self.lr)
        return optimizer



In [ ]:
y_train = y_train[:,2]
class_counts = np.bincount(y_train, minlength=3)
print(class_counts)
weights = len(y_train) / (3 * class_counts)
class_weights = torch.tensor(weights, dtype=torch.float32)
print( class_weights)
y_val = y_val[:,3]
num_inputs = X_train.shape[1]
num_chanels = [20,20,20]
num_levels = 3
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer_ = torch.optim.Adam

tcn_model = TemporalConvNet(num_inputs=num_inputs, num_channels=num_chanels, kernel_size=4, dropout=0.2587, num_outputs=3)

def trainer(model, X_train, y_train, X_val, y_val, criterion, optimizer_, bs=32):
    lightning_model = LightningTCN(model=model, lr=9.880820256268033e-05, criterion=criterion, optimizer_=optimizer_)
    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
        )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    )

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=4)

    early_stop = EarlyStopping(monitor='val loss', mode='min', patience=10)
    checkpoint_callback = ModelCheckpoint(
        dirpath=BASE_DIR/'checkpoints',
        monitor = 'val loss',
        mode = 'min',
        save_top_k =1,
        filename='best_model',
        verbose=True

    )

    trainer = L.Trainer(
        max_epochs=50,
        accelerator='auto',
        devices='auto',
        log_every_n_steps=50,
        callbacks=[early_stop, checkpoint_callback]
    )


    trainer.fit(lightning_model, train_loader, val_loader)
    return trainer, lightning_model

trainer(tcn_model, X_train, y_train, X_val, y_val, criterion, optimizer_)

Epoch 2/49 ━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━ 2252/3779 0:00:36 • 0:00:25 61.27it/s v_num: 31.000 train loss_step:   
                                                                                  0.772 train accuracy_step: 0.594 
                                                                                  val loss: 0.911 val accuracy:    
                                                                                  0.616 train loss_epoch: 0.882    
                                                                                  train accuracy_epoch: 0.637      

In [ ]:
data = pl.scan_parquet(DATA_DIR_FEATURES_TARGETS/'features_targets').sort('timestamp')
target_cols = [f'd{n}' for n in [10,30,60,180]]
X = data.select(pl.all().exclude(target_cols+['timestamp'])).collect().to_numpy()
y = data.select(target_cols).collect().to_numpy()[:,2]

cutoff2 = int( y.shape[0] *0.8 )
X = X[:cutoff2]
y = y[:cutoff2]

def objective(trial, X, y):

    optimizer_ = torch.optim.Adam
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    num_chanels = trial.suggest_categorical('num_channels',([20,20,20], [40,80,160], [10,10,5], [50,100,500]))
    dropout = trial.suggest_float('dropout', 0.2, 0.6)
    kernel_size = trial.suggest_int('kernel_size', 3, 5)
    fold_losses = []
    num_folds = 4
    length = int( y.shape[0] * 0.2 )

    for i in range(num_folds):
        X_train = X[:(i+1)*length]
        y_train = y[:(i+1)*length]

        if i != num_folds-1:
            X_val = X[(i+1)*length:(i+2)*length][180:]
            y_val = y[(i+1)*length:(i+2)*length][180:]
        else:
            X_val = X[(i+1)*length:][180:]
            y_val = y[(i+1)*length:][180:]
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_train = sliding_window(X_train, sql).transpose(0,2,1)
        y_train = y_train[sql-1:]
        X_val = sliding_window(X_val, sql).transpose(0,2,1)
        y_val = y_val[sql-1:]
        class_counts = np.bincount(y_train, minlength=3)
        weights = len(y_train) / (3 * class_counts)
        class_weights = torch.tensor(weights, dtype=torch.float32)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        num_inputs=X_train.shape[1]
        train_ds = TensorDataset(
             torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.long)
        )
        val_ds = TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.long)
         )

        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

        tcn_model = TemporalConvNet(num_inputs=num_inputs, num_channels=num_chanels, kernel_size=kernel_size, dropout=dropout, num_outputs=3)
        lightning_model = LightningTCN(model=tcn_model, lr=lr, criterion=criterion, optimizer_=optimizer_)
        early_stop = EarlyStopping(
             monitor="val loss",
             patience=5,
              min_delta=1e-4,
              mode="min",
              verbose=False,
                 )


        trainer = L.Trainer(
            max_epochs=50,
            accelerator='auto',
            devices=1,
            callbacks=[early_stop],
            enable_progress_bar=False,
            logger=False,
        )
        trainer.fit(lightning_model, train_loader, val_loader)
        fold_loss = trainer.callback_metrics['val loss'].item()
        fold_losses.append(fold_loss)
    avg = sum(fold_losses) / len(fold_losses)
    print(f"[Trial {trial.number}] avg val loss = {avg:.4f} | params={trial.params}")

    return avg

study = optuna.create_study(direction='minimize')
study.optimize(lambda trial: objective(trial, X, y), n_trials=20)

[I 2026-10-22 20:21:18,151] A new study created in memory with name: no-name-d83cdf3b-2606-40f5-a26c-8cf2c3636a97
/tmp/ipykernel_1470/2614019629.py:14: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [20, 20, 20] which is of type list.
  num_chanels = trial.suggest_categorical('num_channels',([20,20,20], [40,80,160], [10,10,5], [50,100,500]))
/tmp/ipykernel_1470/2614019629.py:14: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [40, 80, 160] which is of type list.
  num_chanels = trial.suggest_categorical('num_channels',([20,20,20], [40,80,160], [10,10,5], [50,100,500]))
/tmp/ipykernel_1470/2614019629.py:14: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [10, 10, 5] which is of type list.
  num_chanels = trial.sugg

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ TemporalConvNet  │  270 K │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 270 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 270 K                                                                                                
Total estimated model params size (MB): 1.082                                                                      
Modules in train mode: 58                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.u

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ TemporalConvNet  │  270 K │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 270 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 270 K                                                                                                
Total estimated model params size (MB): 1.082                                                                      
Modules in train mode: 58                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

... (270 outputs left out)

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ TemporalConvNet  │ 10.2 K │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 10.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.2 K                                                                                               
Total estimated model params size (MB): 0.041                                                                      
Modules in train mode: 56                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
INFO: `Trainer.fit` stopped: `max_epochs=50` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
[I 2026-10-23 01:42:52,925] Trial 19 finished with va

[Trial 19] avg val loss = 0.8951 | params={'lr': 1.7335533870600684e-05, 'num_channels': [20, 20, 20], 'dropout': 0.43479236178748853, 'kernel_size': 4}


In [ ]:
params={'lr': 9.880820256268033e-05, 'num_channels': [20, 20, 20], 'dropout': 0.25873117765676085, 'kernel_size': 4}